# Distributed Energy Resources

----

## Inverter Power Flow Modeling 

CIM 17 has introduced detailed modeling of distributed energy resources (DERs) with even more detailed models to be added in the upcoming draft CIM 18. The first set of modeling classes are contained within the Wires and Production packages. The inverter is specified as a `PowerElectronicsConnection` with attributes for the rated voltage and maximum real / reactive / apparent power that can be produced by the inverter. Each `PowerElectronicsConnection` inverter is associated with a single Terminal object on the AC side of the device. No explicit modeling of the DC connectivity is included. Single-phase inverters can be specified by defining each phase component as a `PowerElectronicsConnectionPhase` associated with the overall `PowerElectronicsConnection`. The DC source behind the inverter is specified through Production package as a `PhotoVoltaicUnit`, `BatteryUnit`, or `PowerElectronicsWindUnit`. The minimum and maximum power of each DC source is specified through the source class, as shown in Figure below. Basic inverter control modes are specified as an enumeration.

In [1]:
import os
from cimgraph import utils
from mermaid import Mermaid
import cimgraph.data_profile.cim17v40 as cim
os.environ['CIMG_CIM_PROFILE'] = 'cim17v40'

In [2]:
diagram_text = utils.get_mermaid([cim.RegulatingCondEq, cim.PowerElectronicsConnection,
                                  cim.PowerElectronicsConnectionPhase, cim.PowerElectronicsUnit,
                                  cim.PhotoVoltaicUnit, cim.BatteryUnit, cim.PowerElectronicsWindUnit,
                                  cim.SinglePhaseKind, cim.BatteryStateKind])
Mermaid(diagram_text)

Some examples are discussed below.

In [3]:
import os
from cimgraph.databases import XMLFile
from cimgraph.models import FeederModel
import cimgraph.data_profile.cimhub_2023 as cim
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'

In [4]:
network = FeederModel(connection=XMLFile(filename='../sample_models/ieee13.xml'), container=None)

Example 1: What is the state of charge the inverter with mRID "682AB7A9-4FBF-4204-BDE1-27EAB3425DA0"?

In [5]:
mRID = "682AB7A9-4FBF-4204-BDE1-27EAB3425DA0"
result = None

inverter = network.get_object(mRID=mRID)
for unit in inverter.PowerElectronicsUnit:
    if isinstance(unit, cim.BatteryUnit):
        # State of charge is stored energy divided by capacity
        result = float(unit.storedE) / float(unit.ratedE)

print(result)

0.37037037037037035


Example 2:  What is the charging status of all batteries in the model?

In [6]:
result = []

# batteryState is a BatteryStateKind enumeration (charging / discharging / waiting)
for battery in network.list_by_class(cim.BatteryUnit):
    result.append(str(battery.batteryState))

print(result)

['BatteryStateKind.discharging', 'BatteryStateKind.waiting', 'BatteryStateKind.charging']


Example 3: What is the nominal rated voltage of the inverter named "house"?

In [7]:
name = "house"
result = None

# Inverters use ratedU rather than a BaseVoltage association
for inverter in network.find_by_attribute(cim.PowerElectronicsConnection, 'name', name):
    result = inverter.ratedU

print(result)

208.0


Example 4: What are the rated capacities of the batteries connected to the bus node named "634"?

In [8]:
name = "634"
result = []

# Path: ConnectivityNode -> Terminals -> ConductingEquipment -> PowerElectronicsUnit
for node in network.find_by_attribute(cim.ConnectivityNode, 'name', name):
    for terminal in node.Terminals:
        equipment = terminal.ConductingEquipment
        if isinstance(equipment, cim.PowerElectronicsConnection):
            for unit in equipment.PowerElectronicsUnit:
                if isinstance(unit, cim.BatteryUnit):
                    result.append({'rated_capacity': unit.ratedE})

print(result)

[{'rated_capacity': 200000.0}, {'rated_capacity': 200000.0}]


Example 5: What bus is the battery named "school" connected to?

In [9]:
name = "school"
result = []

# Path: BatteryUnit -> PowerElectronicsConnection -> Terminals -> ConnectivityNode
for battery in network.find_by_attribute(cim.BatteryUnit, 'name', name):
    inverter = battery.PowerElectronicsConnection
    for terminal in inverter.Terminals:
        result.append(terminal.ConnectivityNode.name)

print(result)

['634']


Example 6: What are the names of the batteries connected to the bus node named "634"?

In [10]:
name = "634"
result = []

# Path: ConnectivityNode -> Terminals -> ConductingEquipment -> PowerElectronicsUnit
for node in network.find_by_attribute(cim.ConnectivityNode, 'name', name):
    for terminal in node.Terminals:
        equipment = terminal.ConductingEquipment
        if isinstance(equipment, cim.PowerElectronicsConnection):
            for unit in equipment.PowerElectronicsUnit:
                if isinstance(unit, cim.BatteryUnit):
                    result.append(unit.name)

print(result)

['school', 'batidle']


Example 7: What is the rated output of the inverter connected to the bus node named "634"?

In [11]:
name = "634"
result = []

# ratedS is apparent power (VA). Path: ConnectivityNode -> Terminals -> ConductingEquipment
for node in network.find_by_attribute(cim.ConnectivityNode, 'name', name):
    for terminal in node.Terminals:
        equipment = terminal.ConductingEquipment
        if isinstance(equipment, cim.PowerElectronicsConnection):
            result.append(equipment.ratedS)

print(result)

[300000.0, 100000.0, 100000.0]


Example 8: What is the rated voltage of the inverter connected to battery named "house"?


In [12]:
name = "house"
result = []

# Inverters use ratedU rather than a BaseVoltage association
for battery in network.find_by_attribute(cim.BatteryUnit, 'name', name):
    inverter = battery.PowerElectronicsConnection
    result.append(inverter.ratedU)

print(result)

[208.0]


Example 9: What is the rated capacity of the battery for the inverter with mRID "682AB7A9-4FBF-4204-BDE1-27EAB3425DA0"?


In [13]:
mRID = "682AB7A9-4FBF-4204-BDE1-27EAB3425DA0"
result = []

inverter = network.get_object(mRID=mRID)
for unit in inverter.PowerElectronicsUnit:
    if isinstance(unit, cim.BatteryUnit):
        # ratedE is the rated storage capacity (Wh)
        result.append(unit.ratedE)

print(result)

[13500.0]


Example 10: What is the nominal rated kVA apparent power output of the inverter named "house"?


In [14]:
name = "house"
result = []

for inverter in network.find_by_attribute(cim.PowerElectronicsConnection, 'name', name):
    if inverter.name == name:
        result.append(inverter.ratedS)

print(result)

[5000.0, 5000.0]


Example 11: How much energy is stored in battery for the inverter with mRID "682AB7A9-4FBF-4204-BDE1-27EAB3425DA0"?

In [15]:
mRID = "682AB7A9-4FBF-4204-BDE1-27EAB3425DA0"
result = None

inverter = network.get_object(mRID=mRID)
for unit in inverter.PowerElectronicsUnit:
    if isinstance(unit, cim.BatteryUnit):
        # storedE is the energy currently stored (Wh)
        result = unit.storedE

result

5000.0

Example 12: What is the state of charge the inverter with mRID "682AB7A9-4FBF-4204-BDE1-27EAB3425DA0"?

In [16]:
mRID = "682AB7A9-4FBF-4204-BDE1-27EAB3425DA0"
result = None

inverter = network.get_object(mRID=mRID)
for unit in inverter.PowerElectronicsUnit:
    if isinstance(unit, cim.BatteryUnit):
        # State of charge is stored energy divided by capacity
        result = float(unit.storedE) / float(unit.ratedE)

print(result)

0.37037037037037035


Example 13: What is the largest solar panel in the model?

In [17]:
result = []

# PhotoVoltaicUnit.maxP is the solar panel real power rating
for unit in network.list_by_class(cim.PhotoVoltaicUnit):
    result.append(float(unit.maxP))

result = max(result)

result

300000.0

Example 14: What is the reactive power rating of the inverter named "house"?

In [18]:
name = "house"
result = None

for inverter in network.find_by_attribute(cim.PowerElectronicsConnection, 'name', name):
    if inverter.name == name:
        # minQ = max reactive power absorbed, maxQ = max reactive power injected
        result = {'minQ': inverter.minQ, 'maxQ': inverter.maxQ}

print(result)

{'minQ': -5000.0, 'maxQ': 5000.0}


Example 15: What is the real power rating of the solar panel for inverter named "house"?

In [19]:
name = "house"
result = None

# Path: PowerElectronicsConnection -> PowerElectronicsUnit
for inverter in network.find_by_attribute(cim.PowerElectronicsConnection, 'name', name):
    if inverter.name == name:
        for unit in inverter.PowerElectronicsUnit:
            if isinstance(unit, cim.PhotoVoltaicUnit):
                # maxP is the solar panel real power rating
                result = unit.maxP

print(result)

5000.0


Example 16: Create a new solar panel sized at 5 kW with an inverter with make Enphase6000 sized at 6 kVA connected to bus named 634?

In [20]:
# Create a new inverter and assign its rated apparent power (ratedS, kVA) and name
new_inverter = cim.PowerElectronicsConnection()
new_inverter.uuid()
network.add_to_graph(new_inverter)
new_inverter.ratedS = 6000
new_inverter.name = 'Enphase6000'

# Create a new solar panel and assign its rated real power (maxP, kW)
new_solar = cim.PhotoVoltaicUnit()
new_solar.uuid()
network.add_to_graph(new_solar)
new_solar.maxP = 5000

# Associate the solar panel with the inverter
new_solar.PowerElectronicsConnection = new_inverter
new_inverter.PowerElectronicsUnit.append(new_solar)

# Create a terminal to connect the inverter to the network
new_terminal = cim.Terminal()
new_terminal.uuid()
network.add_to_graph(new_terminal)
new_terminal.ConductingEquipment = new_inverter
new_inverter.Terminals.append(new_terminal)

# Connect the terminal to the bus named 634
for node in network.find_by_attribute(cim.ConnectivityNode, 'name', '634'):
    if node.name == '634':
        new_terminal.ConnectivityNode = node
        node.Terminals.append(new_terminal)

print(new_inverter)
print(new_solar)

{"@id": "e65b6722-55b7-4128-968c-63e1ef738264", "@type": "PowerElectronicsConnection", "name": "Enphase6000", "Terminals": [{"@id": "ff9225db-5eea-473b-857a-f8c8b72b01aa", "@type": "Terminal"}], "ratedS": "6000", "PowerElectronicsUnit": [{"@id": "66df1211-5af9-437c-b0e8-0caae7779604", "@type": "PhotoVoltaicUnit"}]}
{"@id": "66df1211-5af9-437c-b0e8-0caae7779604", "@type": "PhotoVoltaicUnit", "maxP": "5000", "PowerElectronicsConnection": {"@id": "e65b6722-55b7-4128-968c-63e1ef738264", "@type": "PowerElectronicsConnection"}}
